# XGBoost Fraud Detection Experiment
IEEE-CIS Fraud Detection Competition

## 0. Setup & MLflow Configuration

In [ ]:
import os, warnings, json
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from dotenv import load_dotenv
from kaggle_secrets import UserSecretsClient
load_dotenv()

DAGSHUB_TOKEN = UserSecretsClient().get_secret("DAGSHUB_TOKEN")
DAGSHUB_USERNAME = 'delibaaa'
REPO_NAME        = 'DELIBA-ML-ASSIGNMENT-2'

os.environ['MLFLOW_TRACKING_USERNAME'] = DAGSHUB_USERNAME
os.environ['MLFLOW_TRACKING_PASSWORD'] = DAGSHUB_TOKEN

mlflow.set_tracking_uri(f'https://dagshub.com/{DAGSHUB_USERNAME}/{REPO_NAME}.mlflow')
mlflow.set_experiment('XGBoost_Training')
print('Setup complete')

## 1. Data Loading

In [ ]:
DATA_PATH = '/kaggle/input/ieee-fraud-detection'

train = pd.read_csv(f'{DATA_PATH}/train_transaction.csv').merge(
        pd.read_csv(f'{DATA_PATH}/train_identity.csv'), on='TransactionID', how='left')
test  = pd.read_csv(f'{DATA_PATH}/test_transaction.csv').merge(
        pd.read_csv(f'{DATA_PATH}/test_identity.csv'), on='TransactionID', how='left')

print(f'Train: {train.shape} | Test: {test.shape}')
print(f'Fraud rate: {train["isFraud"].mean():.4f}')

## 2. Cleaning

In [ ]:
with mlflow.start_run(run_name='XGBoost_Cleaning'):

    nan_pct = train.isnull().mean()
    high_nan_cols = nan_pct[nan_pct > 0.9].index.tolist()
    train_clean = train.drop(columns=high_nan_cols)
    test_clean  = test.drop(columns=high_nan_cols)

    constant_cols = [c for c in train_clean.columns if train_clean[c].nunique(dropna=False) <= 1]
    train_clean = train_clean.drop(columns=constant_cols)
    test_clean  = test_clean.drop(columns=constant_cols, errors='ignore')

    mlflow.log_param('dropped_high_nan_cols', len(high_nan_cols))
    mlflow.log_param('dropped_constant_cols', len(constant_cols))
    mlflow.log_metric('cols_after_cleaning', train_clean.shape[1])
    print(f'After cleaning: {train_clean.shape}')

## 3. Feature Engineering

In [ ]:
with mlflow.start_run(run_name='XGBoost_Feature_Engineering'):

    def feature_engineering(df):
        df = df.copy()
        df['TransactionHour']   = (df['TransactionDT'] // 3600 % 24).astype(int)
        df['TransactionDay']    = (df['TransactionDT'] // (3600*24)).astype(int)
        df['TransactionWeekDay']= (df['TransactionDay'] % 7).astype(int)
        df['TransactionAmt_log'] = np.log1p(df['TransactionAmt'])
        df['TransactionAmt_decimal'] = df['TransactionAmt'] - df['TransactionAmt'].astype(int)

        if 'isFraud' in df.columns and 'card1' in df.columns:
            card1_fraud_rate = df.groupby('card1')['isFraud'].transform('mean')
            df['card1_fraud_rate'] = card1_fraud_rate

        for col in ['card1', 'card2', 'addr1']:
            if col in df.columns:
                df[f'{col}_freq'] = df[col].map(df[col].value_counts(normalize=True)).fillna(0)

        if 'card1' in df.columns:
            df['card1_amt_mean'] = df.groupby('card1')['TransactionAmt'].transform('mean')

        return df

    train_fe = feature_engineering(train_clean)
    test_fe  = feature_engineering(test_clean)

    mlflow.log_metric('features_after_fe', train_fe.shape[1])
    mlflow.log_param('fe_methods', 'time_features,log_amt,freq_encoding,target_encoding_proxy,aggregations')
    print(f'After FE: {train_fe.shape}')

## 4. Feature Selection

In [ ]:
with mlflow.start_run(run_name='XGBoost_Feature_Selection'):

    TARGET = 'isFraud'
    DROP_COLS = ['TransactionID', 'TransactionDT', TARGET]
    feature_cols = [c for c in train_fe.columns if c not in DROP_COLS]

    cat_cols = train_fe[feature_cols].select_dtypes(include='object').columns.tolist()
    num_cols = train_fe[feature_cols].select_dtypes(exclude='object').columns.tolist()

    train_encoded = train_fe[feature_cols].copy()
    for col in cat_cols:
        train_encoded[col] = train_encoded[col].astype('category').cat.codes
    train_encoded = train_encoded.fillna(-999)

    quick_xgb = xgb.XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        use_label_encoder=False, eval_metric='auc',
        random_state=42, n_jobs=-1, tree_method='hist'
    )
    quick_xgb.fit(train_encoded, train_fe[TARGET])

    importance_df = pd.DataFrame({
        'feature': feature_cols,
        'importance': quick_xgb.feature_importances_
    }).sort_values('importance', ascending=False)

    selected_features = importance_df[importance_df['importance'] > 0]['feature'].tolist()
    print(f'Selected: {len(selected_features)} features')

    importance_df.to_csv('/tmp/xgb_feature_importance.csv', index=False)
    mlflow.log_artifact('/tmp/xgb_feature_importance.csv')
    mlflow.log_param('selected_features_count', len(selected_features))
    mlflow.log_param('selection_method', 'xgb_importance_nonzero')

## 5. Cross Validation (Baseline)

In [ ]:
with mlflow.start_run(run_name='XGBoost_CV_Baseline'):

    X = train_fe[selected_features].copy()
    y = train_fe[TARGET]
    cat_in_sel = [c for c in selected_features if train_fe[c].dtype == 'object']
    for col in cat_in_sel:
        X[col] = X[col].astype('category').cat.codes
    X = X.fillna(-999)

    baseline_params = {
        'n_estimators': 500,
        'max_depth': 6,
        'learning_rate': 0.05,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'use_label_encoder': False,
        'eval_metric': 'auc',
        'random_state': 42,
        'n_jobs': -1,
        'tree_method': 'hist'
    }

    model = xgb.XGBClassifier(**baseline_params)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X, y, cv=skf, scoring='roc_auc', n_jobs=-1)

    print(f'XGB CV AUC: {cv_scores.mean():.5f} (+/- {cv_scores.std():.5f})')
    mlflow.log_params({k:v for k,v in baseline_params.items() if k not in ['use_label_encoder','eval_metric','n_jobs','tree_method']})
    mlflow.log_metric('cv_auc_mean', cv_scores.mean())
    mlflow.log_metric('cv_auc_std', cv_scores.std())

## 6. Hyperparameter Tuning (Optuna)

In [ ]:
with mlflow.start_run(run_name='XGBoost_Tuning'):

    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 200, 800),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'gamma': trial.suggest_float('gamma', 0, 5),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10, log=True),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'use_label_encoder': False, 'eval_metric': 'auc',
            'random_state': 42, 'n_jobs': -1, 'tree_method': 'hist'
        }
        model = xgb.XGBClassifier(**params)
        skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        return cross_val_score(model, X, y, cv=skf, scoring='roc_auc').mean()

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=30, show_progress_bar=True)

    best_params = study.best_params
    best_params.update({'use_label_encoder': False, 'eval_metric': 'auc',
                        'random_state': 42, 'n_jobs': -1, 'tree_method': 'hist'})

    print(f'Best XGB AUC: {study.best_value:.5f}')
    mlflow.log_params({k:v for k,v in best_params.items() if k not in ['use_label_encoder','eval_metric','n_jobs','tree_method']})
    mlflow.log_metric('best_cv_auc', study.best_value)

## 7. Overfit / Underfit Analysis

In [1]:
with mlflow.start_run(run_name='XGBoost_Overfit_Experiment'):
    overfit = xgb.XGBClassifier(
        n_estimators=3000, max_depth=15, learning_rate=0.3,
        use_label_encoder=False, eval_metric='auc', random_state=42, tree_method='hist'
    )
    train_aucs, val_aucs = [], []
    for tr_idx, val_idx in StratifiedKFold(3, shuffle=True, random_state=42).split(X, y):
        overfit.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        train_aucs.append(roc_auc_score(y.iloc[tr_idx], overfit.predict_proba(X.iloc[tr_idx])[:,1]))
        val_aucs.append(roc_auc_score(y.iloc[val_idx], overfit.predict_proba(X.iloc[val_idx])[:,1]))
    print(f'[OVERFIT] Train AUC: {np.mean(train_aucs):.5f} | Val AUC: {np.mean(val_aucs):.5f}')
    mlflow.log_metric('train_auc', np.mean(train_aucs))
    mlflow.log_metric('val_auc', np.mean(val_aucs))
    mlflow.log_metric('overfit_gap', np.mean(train_aucs) - np.mean(val_aucs))
    mlflow.log_param('model_type', 'overfit_intentional')

with mlflow.start_run(run_name='XGBoost_Underfit_Experiment'):
    underfit = xgb.XGBClassifier(
        n_estimators=10, max_depth=2, learning_rate=0.01,
        use_label_encoder=False, eval_metric='auc', random_state=42
    )
    cv_u = cross_val_score(underfit, X, y, cv=StratifiedKFold(3, shuffle=True, random_state=42), scoring='roc_auc')
    print(f'[UNDERFIT] CV AUC: {cv_u.mean():.5f}')
    mlflow.log_metric('cv_auc_mean', cv_u.mean())
    mlflow.log_param('model_type', 'underfit_intentional')

SyntaxError: invalid syntax (<ipython-input-1-4e8f6cc6f110>, line 11)

## 8. Final Pipeline Training & Registration

In [2]:
with mlflow.start_run(run_name='XGBoost_Final') as run:

    X_raw = train_fe[selected_features].copy()
    y_final = train_fe[TARGET]

    cat_feats = X_raw.select_dtypes(include='object').columns.tolist()
    num_feats = X_raw.select_dtypes(exclude='object').columns.tolist()

    preprocessor = ColumnTransformer([
        ('num', SimpleImputer(strategy='constant', fill_value=-999), num_feats),
        ('cat', Pipeline([
            ('imp', SimpleImputer(strategy='constant', fill_value='missing')),
            ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
        ]), cat_feats)
    ])

    final_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', xgb.XGBClassifier(**best_params))
    ])

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    final_cv = cross_val_score(final_pipeline, X_raw, y_final, cv=skf, scoring='roc_auc', n_jobs=-1)
    print(f'Final XGB Pipeline CV AUC: {final_cv.mean():.5f} (+/- {final_cv.std():.5f})')

    final_pipeline.fit(X_raw, y_final)

    mlflow.log_metric('final_cv_auc_mean', final_cv.mean())
    mlflow.log_metric('final_cv_auc_std', final_cv.std())
    mlflow.log_param('n_selected_features', len(selected_features))

    mlflow.sklearn.log_model(
        sk_model=final_pipeline,
        artifact_path='xgb_pipeline',
        registered_model_name='XGBoost_Fraud_Pipeline'
    )
    print(f'XGBoost pipeline registered! Run ID: {run.info.run_id}')

with open('xgb_selected_features.json', 'w') as f:
    json.dump(selected_features, f)

SyntaxError: invalid syntax (<ipython-input-2-6a18a46224a7>, line 24)